## Task-1

In [6]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = client.chat.completions.create(
    model="llama3.2:3b",
    messages=[
        {
            "role": "system",
            "content": "You are a concise support assistant. Classify messages into: billing, bug, feature_request, or other. Answer ONLY with one word."
        },
        {
            "role": "user",
            "content": "My app crashed when I tried to login"
        }
    ],
    temperature=0.1
)

print(response.choices[0].message.content)

Bug


In [7]:
for i in range(3):
    response = client.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": "Classify into billing, bug, feature_request, or other. Answer only one word."},
            {"role": "user", "content": "My app crashed when I tried to login"}
        ],
        temperature=0.1
    )
    print("Run", i+1, ":", response.choices[0].message.content)

Run 1 : Bug
Run 2 : Bug
Run 3 : Bug


In [8]:
for i in range(3):
    response = client.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": "Classify into billing, bug, feature_request, or other. Answer only one word."},
            {"role": "user", "content": "My app crashed when I tried to login"}
        ],
        temperature=1.0
    )
    print("Run", i+1, ":", response.choices[0].message.content)

Run 1 : Bug
Run 2 : Bug
Run 3 : Bug


### Temperature Experiment Results

In this experiment, I tested how temperature affects the model's output by running the same prompt multiple times with different temperature values.

When using a low temperature (0.1), the model produced consistent outputs across all runs. The result was always the same ("bug"), which shows that the model behaves deterministically and selects the most probable answer.

When using a higher temperature (1.0), I expected more variation in the outputs. However, in this specific case, the model still returned the same result ("bug") every time. This is because the input message ("My app crashed when I tried to login") is very clear and strongly associated with the "bug" category.

This demonstrates that temperature controls randomness in the model's responses, but its effect is more noticeable when the input is ambiguous. For clear and obvious cases, the model remains consistent regardless of temperature.

In conclusion, low temperature leads to stable and predictable outputs, while high temperature can introduce variability — but only when multiple interpretations are possible.

## Task-2

In [18]:
tickets = [
    "My app crashes when I open settings",
    "I was charged twice for my subscription",
    "I wish your app had dark mode",
    "The app is very slow and freezes sometimes",
    "Hello, just wanted to say great app!",
    "I can't login to my account",
    "Please add fingerprint login feature"
]

In [19]:
def classify_zero(message):
    response = client.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": "Classify into one of these EXACT labels: billing, bug, feature_request, other. Answer ONLY with one word from the list."},
            {"role": "user", "content": message}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip().lower().replace(" ", "_")

In [20]:
def classify_few(message):
    response = client.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {
                "role": "system",
                "content": "Classify into one of these EXACT labels: billing, bug, feature_request, other. Answer ONLY with one word from the list."
            },
            {
                "role": "user",
                "content": """
Message: I was charged twice
Label: billing

Message: App crashes on login
Label: bug

Message: Please add dark mode
Label: feature_request

Answer ONLY with one of: billing, bug, feature_request, other.
""" + message
            }
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip().lower().replace(" ", "_")

In [21]:
def classify_cot(message):
    response = client.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": "Classify into one of these EXACT labels: billing, bug, feature_request, other. Answer ONLY with one word from the list."},
            {"role": "user", "content": message}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip().lower().replace(" ", "_")

In [22]:
import pandas as pd
results = []

for t in tickets:
    results.append({
        "ticket": t,
        "zero_shot": classify_zero(t),
        "few_shot": classify_few(t),
        "chain_of_thought": classify_cot(t)
    })

import pandas as pd
df = pd.DataFrame(results)
df

,ticket,zero_shot,few_shot,chain_of_thought
0,My app crashes when I open settings,bug,bug,bug
1,I was charged twice for my subscription,bug,billing,bug
2,I wish your app had dark mode,feature_request,feature_request,feature_request
3,The app is very slow and freezes sometimes,bug,bug,bug
4,"Hello, just wanted to say great app!",other,other,other
5,I can't login to my account,bug,bug,bug
6,Please add fingerprint login feature,feature_request,feature_request,feature_request


## Task-3

In [30]:
def classify_json(message):
    response = client.chat.completions.create(
        model="llama3.2:3b",
        messages=[
            {
                "role": "system",
                "content": """You are a classifier.

Return ONLY valid JSON in this format:
{
  "label": "billing | bug | feature_request | other",
  "confidence": number between 0 and 1,
  "reason": "short explanation"
}

Label rules:
- billing → payments, charges, subscriptions
- bug → crashes, errors, not working
- feature_request → asking for new features
- other → anything else
"""
            },
            {
                "role": "user",
                "content": message
            }
        ],
        temperature=0.1
    )

    return response.choices[0].message.content

In [31]:
import json

def safe_classify(message):
    raw = classify_json(message)

    try:
        data = json.loads(raw)

        label = data.get("label", "").lower()
        confidence = data.get("confidence", 0)

        valid_labels = ["billing", "bug", "feature_request", "other"]

        if label not in valid_labels:
            label = "other"

        if not (0 <= confidence <= 1):
            confidence = 0.5

        return {
            "label": label,
            "confidence": confidence,
            "reason": data.get("reason", "")
        }

    except:
        #  BAD OUTPUT HANDLE
        return {
            "label": "other",
            "confidence": 0.0,
            "reason": "Invalid JSON from model"
        }

In [32]:
print(safe_classify("My app crashed when I login"))
print(safe_classify("I was charged twice"))

{'label': 'bug', 'confidence': 0.9, 'reason': 'App crashed during login process'}
{'label': 'other', 'confidence': 0.8, 'reason': 'Unexpected duplicate charge detected'}


### Structured Output (JSON)

In this task, the classifier was updated to return JSON:

{
  "label": "billing | bug | feature_request | other",
  "confidence": 0.0,
  "reason": "short string"
}

The output was parsed in Python and validated:
- label must be valid
- confidence must be between 0 and 1
- invalid JSON is handled with a fallback ("other", 0.0)


### Ollama vs Reliability

When using the local Ollama model (llama3.2:3b):
- Sometimes returned invalid JSON
- Added extra text or wrong format
- Caused parsing errors

Compared to that, stronger models follow JSON format more reliably.



### Conclusion

Structured output makes results usable in real systems.  
However, smaller models like Ollama are less consistent, so validation and error handling are required.